# Phase 6 — Experimental Feedback Integration

The final phase closes the loop between computation and experiment.
After wet-lab characterization of the top candidates, we:

1. Ingest SPR kinetics, thermal shift, and LFA prototype data
2. Correlate each computational metric with experimental outcomes
3. Recalibrate scoring weights to maximize predictive accuracy
4. Re-rank all prior candidates to find "hidden gems"
5. Optionally launch additional design rounds with improved scoring

This notebook uses **simulated experimental data** to demonstrate
the analysis pipeline.

In [ ]:
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from nanolfa.core.calibration import (
    ExperimentalData,
    compute_correlation,
    recalibrate_weights,
    find_hidden_gems,
    save_calibration,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

OUTPUT_DIR = Path('../data/results/calibration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Simulate Experimental Data

In production, this data comes from SPR/BLI, thermal shift assays,
and LFA prototype testing. Here we generate correlated synthetic data
to demonstrate the analysis.

In [ ]:
rng = np.random.default_rng(42)
n_candidates = 20

# Simulated computational scores (normalized 0-1)
comp_scores = {}
for i in range(n_candidates):
    cid = f'candidate_{i:03d}'
    base = rng.uniform(0.3, 0.9)
    comp_scores[cid] = {
        'iptm': float(np.clip(base + rng.normal(0, 0.1), 0, 1)),
        'plddt_interface': float(np.clip(base + rng.normal(0, 0.08), 0, 1)),
        'shape_complementarity': float(np.clip(base * 0.8 + rng.normal(0, 0.1), 0, 1)),
        'binding_energy': float(np.clip(base + rng.normal(0, 0.12), 0, 1)),
        'buried_surface_area': float(np.clip(base * 0.7 + rng.normal(0, 0.1), 0, 1)),
        'developability': float(np.clip(0.7 + rng.normal(0, 0.1), 0, 1)),
    }

# Simulated experimental data (correlated with some comp metrics)
exp_data = []
for i in range(n_candidates):
    cid = f'candidate_{i:03d}'
    cs = comp_scores[cid]
    
    # KD correlates with binding_energy and iptm (with noise)
    log_kd_pred = -7 - 2.5 * cs['binding_energy'] - 1.5 * cs['iptm'] + rng.normal(0, 0.5)
    kd = 10 ** log_kd_pred
    
    # Tm correlates with developability
    tm = 55 + 25 * cs['developability'] + rng.normal(0, 3)
    
    # LFA SNR correlates with iptm and shape_complementarity
    snr = 2 + 15 * cs['iptm'] + 8 * cs['shape_complementarity'] + rng.normal(0, 2)
    
    exp_data.append(ExperimentalData(
        candidate_id=cid,
        kd=float(kd),
        kon=float(1e5 * (1 + rng.exponential(2))),
        koff=float(kd * 1e5 * (1 + rng.exponential(2))),
        tm_celsius=float(tm),
        tagg_celsius=float(tm - 12 + rng.normal(0, 2)),
        signal_noise_ratio=float(np.clip(snr, 1, 30)),
        detection_limit_ng_ml=float(np.clip(100 / snr, 1, 50)),
    ))

print(f'Generated {n_candidates} candidates with simulated experimental data')
print(f'\n{"Candidate":<15} {"KD (nM)":>10} {"Tm (C)":>8} {"SNR":>6}')
print('-' * 42)
for d in sorted(exp_data, key=lambda x: x.kd or 1e10)[:10]:
    kd_nm = (d.kd or 0) * 1e9
    print(f'{d.candidate_id:<15} {kd_nm:>10.1f} {d.tm_celsius or 0:>8.1f} {d.signal_noise_ratio or 0:>6.1f}')

## 2. Correlation Analysis

Which computational metrics best predict experimental binding affinity?
We compute Pearson and Spearman correlations for each metric against
log(KD), then visualize the scatter plots.

In [ ]:
# Compute correlations for each metric vs log(KD)
metric_names = ['iptm', 'plddt_interface', 'shape_complementarity',
                'binding_energy', 'buried_surface_area', 'developability']

correlations = []
for metric in metric_names:
    predicted = [comp_scores[d.candidate_id][metric] for d in exp_data]
    experimental = [d.log_kd for d in exp_data if d.log_kd is not None]
    cids = [d.candidate_id for d in exp_data]
    
    corr = compute_correlation(
        predicted, experimental, cids,
        metric_name=metric, experimental_metric='log_KD',
    )
    correlations.append(corr)

# Sort by absolute correlation
correlations.sort(key=lambda c: abs(c.pearson_r), reverse=True)

print(f'{"Metric":<25} {"Pearson r":>10} {"Spearman ρ":>11} {"R²":>8} {"p-value":>10}')
print('-' * 68)
for c in correlations:
    sig = '***' if c.p_value < 0.001 else '**' if c.p_value < 0.01 else '*' if c.p_value < 0.05 else ''
    print(f'{c.metric_name:<25} {c.pearson_r:>10.3f} {c.spearman_rho:>11.3f} {c.r_squared:>8.3f} {c.p_value:>9.4f} {sig}')

In [ ]:
# Scatter plots: each metric vs log(KD)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, corr in zip(axes.flat, correlations):
    ax.scatter(corr.predicted_values, corr.experimental_values,
               s=50, alpha=0.7, edgecolors='white', c='#2196F3')
    
    # Regression line
    x_range = np.linspace(min(corr.predicted_values), max(corr.predicted_values), 100)
    ax.plot(x_range, corr.slope * x_range + corr.intercept,
            'r--', alpha=0.7, linewidth=2)
    
    ax.set_xlabel(f'{corr.metric_name} (predicted)', size=10)
    ax.set_ylabel('log(KD)', size=10)
    ax.set_title(
        f'{corr.metric_name}\nr={corr.pearson_r:.3f}, R²={corr.r_squared:.3f}',
        size=11,
    )

plt.suptitle('Computational Metrics vs Experimental log(KD)', size=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Weight Recalibration

The original scoring weights were set by expert judgment before any
experimental data existed. Now we can update them based on which
metrics actually predict binding affinity.

Ridge regression learns optimal weights while regularizing to prevent
overfitting on a small dataset.

In [ ]:
# Current weights from config
current_weights = {
    'iptm': 0.25,
    'plddt_interface': 0.20,
    'shape_complementarity': 0.15,
    'binding_energy': 0.20,
    'buried_surface_area': 0.10,
    'developability': 0.10,
}

recal = recalibrate_weights(
    correlations, current_weights, method='correlation',
)

print(f'{"Metric":<25} {"Old Weight":>10} {"New Weight":>10} {"Change":>10}')
print('-' * 58)
for metric in current_weights:
    old = recal.old_weights[metric]
    new = recal.new_weights[metric]
    change = recal.weight_changes[metric]
    arrow = '↑' if change > 0.01 else '↓' if change < -0.01 else '='
    print(f'{metric:<25} {old:>10.3f} {new:>10.3f} {change:>+9.3f} {arrow}')

In [ ]:
# Visualize weight changes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = list(current_weights.keys())
old_w = [recal.old_weights[m] for m in metrics]
new_w = [recal.new_weights[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, old_w, width, label='Original', color='#9E9E9E', alpha=0.8)
ax.bar(x + width/2, new_w, width, label='Recalibrated', color='#2196F3', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in metrics], size=9)
ax.set_ylabel('Weight', size=12)
ax.set_title('Scoring Weights: Before vs After Calibration', size=13)
ax.legend()

# Correlation bar chart
ax = axes[1]
corr_values = [abs(c.pearson_r) for c in correlations]
corr_names = [c.metric_name for c in correlations]
colors = ['#4CAF50' if r > 0.5 else '#FF9800' if r > 0.3 else '#F44336' for r in corr_values]
ax.barh(corr_names, corr_values, color=colors, height=0.6)
ax.axvline(x=0.5, color='green', linestyle='--', alpha=0.5, label='Strong (|r|>0.5)')
ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.5, label='Moderate (|r|>0.3)')
ax.set_xlabel('|Pearson r| with log(KD)', size=12)
ax.set_title('Metric-Experiment Correlation Strength', size=13)
ax.legend(fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'weight_recalibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Hidden Gems

Re-ranking with recalibrated weights may reveal candidates that were
computationally undervalued by the original scoring but have strong
experimental data. These are worth rescuing from the reject pile.

In [ ]:
gems = find_hidden_gems(
    comp_scores, recal.old_weights, recal.new_weights,
    exp_data, top_n=10,
)

if gems:
    print(f'{"Candidate":<15} {"Old Rank":>9} {"New Rank":>9} {"Improvement":>12} {"KD (nM)":>10} {"SNR":>6}')
    print('-' * 65)
    for g in gems:
        kd_nm = g.experimental_kd * 1e9 if g.experimental_kd else 0
        snr = g.experimental_snr or 0
        print(
            f'{g.candidate_id:<15} {g.old_rank:>9} {g.new_rank:>9} '
            f'{g.rank_improvement:>+12} {kd_nm:>10.1f} {snr:>6.1f}'
        )
else:
    print('No hidden gems found.')

In [ ]:
# Rank change visualization
if gems:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for g in gems:
        color = '#4CAF50' if g.rank_improvement > 3 else '#FF9800'
        ax.annotate(
            '', xy=(1, g.new_rank), xytext=(0, g.old_rank),
            arrowprops=dict(arrowstyle='->', color=color, lw=2, alpha=0.6),
        )
        ax.text(-0.05, g.old_rank, g.candidate_id, ha='right', va='center', size=8)
        ax.text(1.05, g.new_rank, f'#{g.new_rank}', ha='left', va='center', size=8)
    
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(max(g.old_rank for g in gems) + 2, 0)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Original Ranking', 'Recalibrated Ranking'], size=12)
    ax.set_ylabel('Rank (lower = better)', size=12)
    ax.set_title('Hidden Gems: Rank Changes After Recalibration', size=13)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'hidden_gems.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Save and Apply Calibration

The recalibrated weights are saved as a JSON file. To apply them,
update the `weights` section in `configs/scoring.yaml` and re-run
the design loop if desired.

In [ ]:
cal_path = save_calibration(recal, correlations, OUTPUT_DIR)
print(f'Calibration saved to {cal_path}')
print(f'\nTo apply:')
print(f'  1. Open configs/scoring.yaml')
print(f'  2. Update the weights section with values from {cal_path}')
print(f'  3. Set calibration.enabled: true')
print(f'  4. Optionally re-run Phase 3 with improved scoring')

## Summary

Phase 6 closes the computational-experimental loop:

1. **Ingested** SPR kinetics, thermal shift, and LFA prototype data
2. **Correlated** each computational metric with experimental log(KD)
3. **Recalibrated** scoring weights based on which metrics actually predict binding
4. **Identified** hidden gems that were computationally underranked

With recalibrated scoring, subsequent design rounds will be more
efficient — the pipeline learns from its own experimental feedback.

---

**Pipeline complete.** The top candidates from this analysis are ready
for scale-up production, gold conjugation, and integration into
prototype lateral flow assays for PdG and E3G detection in urine.